# 14. PAD-UFES-20: pacientes separados y metadata ajustada en train

Flujo nuevo e independiente. Primero preparar/verificar y probar; entrenamiento completo desactivado. Copia los scripts de esta carpeta a Drive, sin `.venv` ni `__pycache__`. Consulta README.md.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import subprocess, sys
PROJECT_DIR = Path("/content/drive/MyDrive/PFC1 - Noemi")
CODE_DIR = PROJECT_DIR / "experiments" / "patient_split_v1"
DATA_DIR = PROJECT_DIR / "dataset" / "PAD-UFES-20"
OUTPUT_DIR = CODE_DIR / "artifacts_colab_v1"
assert (CODE_DIR / "run_experiment.py").exists(), "Copia primero los scripts nuevos a Drive"
assert (DATA_DIR / "metadata.csv").exists()

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(CODE_DIR / "requirements.txt")], check=True)

## Preparar o verificar, sin entrenar
Las versiones instaladas deben cargarse en una sesión limpia. Si Colab solicita reinicio tras la instalación, reinicia y ejecuta de nuevo montaje y configuración antes de continuar.

In [ ]:
def cli(command, *extra):
    subprocess.run([sys.executable, str(CODE_DIR / "run_experiment.py"), command,
                    "--data-dir", str(DATA_DIR), "--output", str(OUTPUT_DIR), *extra], check=True)
cli("verify" if (OUTPUT_DIR / "protocol.json").exists() else "prepare")

In [ ]:
import pandas as pd, json
display(pd.read_csv(OUTPUT_DIR / "split_summary.csv"))
state = json.loads((OUTPUT_DIR / "splits" / "seed_42" / "preprocessor.json").read_text())
display(pd.DataFrame(state["numeric"]).T)
subprocess.run([sys.executable, str(CODE_DIR / "test_protocol.py")], check=True)

## Prueba técnica pequeña
Cuatro entrenamientos de dos épocas, 128 imágenes train y 64 de validación. No evalúa test. Usa un RUN_ID distinto si la prueba ya existe.

In [ ]:
SMOKE_RUN_ID = "smoke_colab_01"
cli("smoke", "--run-id", SMOKE_RUN_ID)

## Experimento completo — ejecución explícita
90 entrenamientos (9 modelos × 2 tareas × 5 particiones). Cambia RUN_FULL a True únicamente después de revisar los splits y la prueba. ADD_FACTORIAL añade 10 controles arquitectura × pérdida en binario.

In [ ]:
RUN_FULL = False
ADD_FACTORIAL = False
FULL_RUN_ID = "paper_baseline_v1"
if RUN_FULL:
    extra = ["--factorial"] if ADD_FACTORIAL else []
    cli("full", "--run-id", FULL_RUN_ID, *extra)
else:
    print("Entrenamiento completo pendiente: revisa los artefactos antes de activar RUN_FULL.")

In [ ]:
full_dir = OUTPUT_DIR / "runs" / FULL_RUN_ID
if (full_dir / "summary.csv").exists():
    display(pd.read_csv(full_dir / "summary.csv"))
    display(pd.read_csv(full_dir / "metrics_per_seed.csv"))